# Fleet Sizing from Simulated Demand

## Model Notes

**Principle:** Construct plausible Y2–Y5 calendars by perturbing the Y1 calendar, then re-run the peak simulation on each constructed calendar so the peak emerges from structure.

**Inputs**
- Y1 event list: for each event — on-hire start, on-hire end, win rate, and asset quantity by type (the footprint that creates overlaps).
- Event growth rate (e.g. 10%/yr) → applied to *event count*.
- Seasonal placement distribution: the empirical month-weights from Y1 (heavily Oct–Jan), expressed as a probability of a new event landing in each month.
- Number of placement draws per year (e.g. 200–500 constructed calendars).

**Step-by-step**
1. **Carry forward the base.** Y2 base calendar = the Y1 events (same dates, durations, quantities). This is the recurring spine.
2. **Compute new-event count.** New events in Y2 = round(Y1 count × growth) − Y1 count. (Y1=12 → Y2≈13 → 1 new event; Y3≈15 → 2 new; etc.) Round to whole events — you can't add 1.2 events.
3. **Characterise the "typical" new event.** Give each new event a duration and asset footprint drawn from the Y1 distribution (e.g. sample an existing event's profile, or use the median / typical-duration archetype). Don't invent an outlier unless you have a reason.
4. **Place each new event stochastically.** For each new event, draw a start month from the seasonal distribution (so most land Oct–Jan), then a start day within that month. This is the one random element.
5. **Assemble one constructed calendar** = base spine + placed new events.
6. **Re-run the peak simulation on it.** On that constructed calendar, run the win-rate Monte Carlo exactly as in Y1: coin-flip each event, walk the calendar day-by-day, record the peak. Repeat across trials → a peak distribution *for that one placement*.
7. **Repeat placement draws.** Do steps 4–6 across many placements. Each placement yields its own peak distribution; pool them. The pooled distribution now reflects *both* win uncertainty *and* schedule-placement uncertainty.
8. **Read the percentile.** Take the P85 (or chosen coverage percentile) of the *pooled* distribution → required fleet for Y2.
9. **Roll forward.** Y3 base = Y2 base + Y2's new events (now part of the recurring spine); repeat steps 2–8 for Y3, Y4, Y5.

**Outputs (per year)**
- A peak demand distribution (not a point) by asset type.
- Required owned fleet at the chosen percentile.
- Capex = max(0, required fleet − fleet held), stepped only when the re-derived peak breaches the owned line.

**Key property:** the peak now moves in steps — a new event either collides with the Oct–Jan cluster (peak jumps) or lands in a trough (peak flat) — because it's read off a calendar, not scaled. The roll-forward propagates the spine so each year inherits the prior year's structure.

```txt
Y1 list ──► [carry forward spine] ──► add round(count×growth) new events
                                          │
                          draw month (seasonal wts) + day  ── one placement
                                          │
                              assemble constructed calendar
                                          │
                          win-rate Monte Carlo on that calendar ─► peak dist (this placement)
                                          │
              repeat over many placements ─► POOLED peak dist
                                          │
                              read P85 ─► required fleet ─► capex (stepped)
                                          │
              spine += this year's new events ─► next year
```

**Model flaws**

1) **Placement uncertainty collapses at the year boundary**: Within a year (step 4), new events are placed stochastically (re-drawn each placement). But at the end of the year, the roll-forward commits the new events to the spine via a single draw — one random placement — which then becomes fixed history for all future years. The cost is that later-year peaks are conditioned on one arbitrary placement history rather than the distribution of histories. The current structure understates compounding uncertainty (i.e does not carry full placement uncertainty forward), but is a pragmatic way to avoid combinatorial explosion of structuring an events tree ($\text{Calendar Placements}^7$ by year 8).
2) **Re-signing is treated as independent: event acquisition vs renewals**: Every year, the model re-flips every event in the spine at its static win rate. So the model assumes your probability of resigning an event in Y2 has the same probability of winning as winning it cold, even though in reality a recurring event you delivered last year should re-sign at a much higher rate (incumbency, relationship, proven delivery). The model treats every year as a fresh tender for the entire portfolio. That systematically understates the recurring base and therefore understates the stable, ownable peak — it makes demand look more volatile and less recurring than it really is. For a business whose whole thesis is "these events repeat and we grow the relationship," that's a meaningful conservatism in the wrong direction.
3) **Win probabilities stay static through market maturation**: As reputation and the market grow, cold acquisition rates might drift up. This is a gentle annual escalator on the base win rate. However, it has mild effects, can be easily overstated, and this rate is bound to plateau due to competition.
4) **Events can be correlated and lack independence**: The simulation flips each event independently, but real wins can be correlated — the same organizers and venues drive several events, so a good relationship year lifts several at once. This fattens the tails, and our P85 coverage is somewhat higher than the independent simulation reports. Modeling it properly means introducing a shared latent factor (a "good year / bad year" draw per organizer that nudges all their events' win draws together) rather than independent coin-flips. However this is the hardest to parametrize and model and can be avoided.

**Caution**
Each added dependency makes the model less transparent and adds parameters you can't observe well (what is the renewal rate for Saadiyat Nights? It's anyone's guess). The current independent model is wrong in known, conservative directions — it understates the recurring base (a) and understates the tail (c). Knowing the direction of the error is itself useful. Before adding machinery, decide whether a defensible conservative bias is actually a problem for the decision you're making, or whether it's a feature you can state plainly: "this sizing assumes we re-win nothing automatically and events are independent — both push the number down, so it's a floor, not a forecast." 

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'

## Inputs

In [ ]:
ASSETS = ["Cabins", "Buggies", "Barriers", "Fencing", "Flooring"]
BILLING = ["daily","daily","monthly","monthly","monthly"]
COVERAGE_PCT = np.array([0.85, 0.5, 0.85, 0.85, 0.85]) # Percentile to own

# Analysis window
YEAR = 8

# Event growth rate
GROWTH = 0.10

# Simulation scale
CALENDAR_PLACEMENTS = 300 # more schedule scenarios
TRIALS = 4_000 # tighter win-rate distribution

# Months in season order (Oct..Sep) and their lengths, used to place new events.
MONTHS = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MONTH_LEN = [31, 30, 31, 31, 28, 31, 30, 31, 30, 31, 31, 30]
MONTH_START = np.cumsum([0] + MONTH_LEN[:-1])  # day offset of each month's 1st

SEED=12
RNG = np.random.default_rng(SEED)

### Y1 Event Calendar

In [ ]:
# (name, win rate, hire start, hire end, cabin, buggies, barriers, fencing, flooring)
events = [
    ('Saadiyat Nights',0.5,1,157,10,10,0,0,280),
    ('BRED X',0.5,22,43,8,0,0,0,0),
    ('MOTN DHA',0.5,34,56,8,6,350,550,0),
    ('No Art',0.5,36,57,24,6,0,0,200),
    ('Tanweer Festival',0.5,44,56,0,8,900,50,200),
    ('MOTN AAN',0.5,40,66,18,6,350,550,0),
    ('Keinemusik',0.5,57,69,29,8,714,1180,178),
    ('MOTN AUH',0.5,50,102,20,10,700,1100,200),
    ('Yas Winterfest',0.5,56,88,6,4,644,760,60),
    ('Animenia',0.5,118,140,8,4,288,192,0),
    ('DAZ',0.5,170,200,15,4,775,225,60),
    ('BRED',0.5,196,210,22,6,1500,500,0),
]
event_names = [e[0] for e in events]
win_rate = np.array([r[1] for r in events])
start = np.array([r[2] for r in events])
end = np.array([r[3] for r in events])
qty = np.array([r[4:] for r in events], dtype=float)

### Seasonal Weights

In [ ]:
days = np.arange(365)
on_hire = (days[None, :] >= start[:, None]) & (days[None, :] <= end[:, None])
on_hire = on_hire.astype(np.float32)

# presence/occupancy weight (not an event-start weight)
# Event count per month (regardless of duration. provides a coarse seasonal shape)
counts = np.array([np.count_nonzero(on_hire[:,s:s+l].sum(axis=1)) for s, l in zip(MONTH_START, MONTH_LEN)])
y1_weights = counts / counts.sum()

for month, count, weight in zip(MONTHS, counts, y1_weights):
    print(f"{month:>2}: {count:2d} events  ({weight:6.1%})")

In [ ]:
# Day-weighted occupancy per month
counts = np.array([on_hire[:,s:s+l].sum(axis=0).sum() for s, l in zip(MONTH_START, MONTH_LEN)], dtype=int)
y1_weights = counts / counts.sum()

for month, count, weight in zip(MONTHS, counts, y1_weights):
    print(f"{month:>2}: {count:>3} event-days  ({weight:6.1%})")

In [ ]:
# Seasonal weights for placing NEW events (Oct..Sep) — concentrated Oct–Jan.
weights = np.array([4, 14, 8, 6, 4, 2, 1, 0, 0, 0, 0, 0])
MONTH_WEIGHTS = weights / weights.sum()

for month, weight in zip(MONTHS, MONTH_WEIGHTS):
    print(f"{month:>2}: {weight:6.1%}")

### Event Duration Archetype

In [ ]:
dur = end - start + 1
mean_dur = np.mean(dur, axis=0)
median_dur = np.median(dur, axis=0)

In [ ]:
print(f"""Mean Dur.: {mean_dur:.0f}days
Median Dur.: {median_dur:.0f}days

Min. Dur.: {np.min(dur)}days
Max. Dur.: {np.max(dur)}days
""")

### Asset Requirement Archetype

In [ ]:
mean_qty = np.mean(qty, axis=0)
median_qty = np.median(qty, axis=0)

width = max(len(asset) for asset in ASSETS)
print(f"{'Asset':<{width}}  {'Mean':>10}  {'Median':>10}")
print(f"{'-' * width}  {'-' * 10}  {'-' * 10}")
for asset, mean, median in zip(ASSETS, mean_qty, median_qty):
    print(
        f"{asset:<{width}}  "
        f"{mean:10.1f}  "
        f"{median:10.1f}"
    )

### Event Archetype

In [ ]:
# (duration, win rate, ["cabins", "buggies", "barriers", "fencing", "flooring"])
ARCHETYPE = (25, 0.5, np.array([13, 6, 500, 400, 100], dtype=float))

### Horizon

In [ ]:
# Calendar window in days for peak demand simulation 
# Extend to cover every event's full on-hire span

last_season_month = np.nonzero(weights)[0][-1] + 1
# Last day of the last active month + duration of archetype event
max_horizon = MONTH_START[last_season_month] - 1 + ARCHETYPE[0]
max_horizon

In [ ]:
# Select max horizon of 1 year or clipped off empty trailing columns that can never contain the peak
HORIZON = max(max_horizon, end.max()+1) # or 365

## Simulation

### Peak Simulation

In [ ]:
# on_hire[i, d]  = 1 if event i is on hire on day d
days = np.arange(HORIZON)
on_hire = (days[None, :] >= start[:, None]) & (days[None, :] <= end[:, None])
on_hire = on_hire.astype(np.float32)

offsets = np.arange(len(events), 0, -1)

fig, ax = plt.subplots(figsize=(10, 3))

for i in range(len(events)):
    ax.plot(
        days,
        np.where(on_hire[i], offsets[i], np.nan),
        lw=4,
    )

ax.set_xlabel("Day")
ax.set_yticks(offsets)
ax.set_yticklabels(event_names, fontsize=8)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="x", alpha=0.2)

plt.show()

In [ ]:
signed = (RNG.random((TRIALS, len(start))) < win_rate).astype(np.float32)

results = np.unique(signed.sum(axis=1), return_counts=True)

width = max(results[1]) // 100
print("Count of total signed events across trials:")
print()
print(f"{'# Events':<10}  {'Trails':<{width}}")
print(f"{'-' * 10}  {'-' * width}")
for event, count in zip(*results):
    print(f"{event:<10.0f} {count:<{width}}")

In [ ]:
results = signed.sum(axis=0)

print("Count of signed events across trials:")
print()
print(f"{'Events':<20}  {'Trails':<10}")
print(f"{'-' * 20}  {'-' * 10}")
for event, count in zip(event_names, results):
    print(f"{event:<20} {count:<10.0f}")

In [ ]:
duration = end - start + 1
months = np.ceil(duration / 30)
weeks = np.ceil(duration / 7)

peaks = np.empty((TRIALS, len(ASSETS)), dtype=np.float32)
asset_days = np.empty((TRIALS, len(ASSETS)), dtype=np.float32)
for a in range(len(ASSETS)):
    units_per_day = on_hire * qty[:, a][:, None]  # (n_events x HORIZON)
    demand = signed @ units_per_day  # (TRIALS x HORIZON)
    peaks[:, a] = demand.max(axis=1)
    if BILLING[a] == "monthly":
        billed_per_event = months * 30 * qty[:, a]
        asset_days[:, a] = signed @ billed_per_event
    else:
        asset_days[:, a] = demand.sum(axis=1)    

print(asset_days.mean(axis=0))

In [ ]:
def simulate_demand(start, end, qty, win_rate):
    """Run the win-rate Monte Carlo on ONE calendar.

    For each TRIALS, flip a weighted coin per event; for the events that 'sign',
    walk the calendar and record (a) the highest concurrent demand — the peak,
    used to size the fleet — and (b) the billable asset-days — the volume, used for revenue.

    Returns
    -------
    peaks      : (TRIALS, n_assets) peak concurrent units per asset, per trial.
                 Always actual concurrency (slab billing never affects the peak).
    asset_days : (TRIALS, n_assets) billable asset-days per asset, per trial.
                 Daily-billed assets count actual days; slab-billed assets bill
                 by the ceiling of duration / slab-length, per event.

    Index maps
    ----------
      on_hire[i, d] = 1 if event i is on hire on day d        (n_events x HORIZON)
      signed[t, i]  = 1 if event i won in trial t             (TRIALS x n_events)
      demand[t, d]  = units on hire on day d in trial t       (TRIALS x HORIZON)
    """
    n_events = len(start)

    # on_hire[i, d] : is event i on hire on day d?            (n_events x HORIZON)
    days = np.arange(HORIZON)
    on_hire = (days[None, :] >= start[:, None]) & (days[None, :] <= end[:, None])
    on_hire = on_hire.astype(np.float32)

    # signed[t, i] : did event i sign in trial t?             (TRIALS x n_events)
    signed = (RNG.random((TRIALS, n_events)) < win_rate).astype(np.float32)

    # Per-event billable slabs, rounded UP per event (never round the total).
    duration = end - start + 1                      # (n_events,) on-hire days
    months = np.ceil(duration / 30)                 # monthly slabs per event
    weeks  = np.ceil(duration / 7)                  # weekly slabs per event

    peaks      = np.empty((TRIALS, len(ASSETS)), dtype=np.float32)
    asset_days = np.empty((TRIALS, len(ASSETS)), dtype=np.float32)

    for a in range(len(ASSETS)):
        # Daily concurrency for this asset — daily demand = signed events' units summed by day
        units_per_day = on_hire * qty[:, a][:, None]   # (n_events x HORIZON)
        demand = signed @ units_per_day                # (TRIALS x HORIZON)
        peaks[:, a] = demand.max(axis=1)               # peak = busiest single day

        # Billable volume — exactly one branch runs per asset.
        if BILLING[a] == "monthly":
            # bill each event ceil(days/30) months, expressed back as asset-days
            billed_per_event = months * 30 * qty[:, a]      # (n_events,)
            asset_days[:, a] = signed @ billed_per_event    # win rate applied via `signed`
        elif BILLING[a] == "weekly":
            billed_per_event = weeks * 7 * qty[:, a]
            asset_days[:, a] = signed @ billed_per_event
        else:  # "daily"
            asset_days[:, a] = demand.sum(axis=1)           # actual asset-days

    return peaks, asset_days, signed

### Add New Events

In [ ]:
def add_new_events(start, end, qty, win_rate, n_new, commit_mode=False):
    """Append n_new ARCHETYPE events to a calendar, placing each by season.

    Each new event takes the ARCHETYPE's duration, win rate, and asset
    quantities, and is dropped onto the calendar at a randomly chosen month
    (weighted toward the busy Oct–Jan window) on a random day within it.

    Two placement modes
    --------------------
    commit_mode=False (simulation):
        Months are drawn from the FULL seasonal distribution (MONTH_WEIGHTS).
        Used inside simulate_year to explore many possible placements of the
        same new event — this is the schedule-placement uncertainty.
    commit_mode=True (roll-forward commit):
        Months are drawn from only the TOP 3 highest-weighted months. Used once
        per year to freeze this year's new events into the spine, so later years
        inherit a realistic (busy-season) placement rather than a tail one.

    Parameters
    ----------
    start, end : (n_events,) int arrays — on-hire day range, inclusive.
    qty        : (n_events, n_assets) — units of each asset per event.
    win_rate   : (n_events,) — per-event probability of signing.
    n_new      : int — number of new events to add (0 returns inputs unchanged).
    commit_mode: bool — placement mode (see above).

    Returns
    -------
    start, end, qty, win_rate : the inputs with n_new rows appended.
    """
    # Nothing to add — return the spine untouched.
    if n_new == 0:
        return start, end, qty, win_rate

    arch_dur, arch_win_rate, arch_qty = ARCHETYPE
    new_start, new_end, new_win_rate, new_qty = [], [], [], []

    for _ in range(n_new):
        if commit_mode:
            # Restrict placement to the 3 busiest months, renormalised to sum to 1.
            top3 = np.argsort(MONTH_WEIGHTS)[-3:]
            p = np.zeros_like(MONTH_WEIGHTS)
            p[top3] = MONTH_WEIGHTS[top3]
            p /= p.sum()
        else:
            # Full seasonal distribution.
            p = MONTH_WEIGHTS

        m = RNG.choice(len(MONTHS), p=p)                     # pick a month (by position)
        day = MONTH_START[m] + RNG.integers(0, MONTH_LEN[m])  # pick a day within that month

        new_start.append(day)
        new_end.append(day + arch_dur - 1)                   # span = archetype duration
        new_qty.append(arch_qty)
        new_win_rate.append(arch_win_rate)

    # Append the new events to each parallel array.
    start    = np.concatenate([start, new_start])
    end      = np.concatenate([end, new_end])
    qty      = np.vstack([qty, new_qty])
    win_rate = np.concatenate([win_rate, new_win_rate])
    return start, end, qty, win_rate

### Simulate Year

In [ ]:
def simulate_year(start, end, qty, win_rate, n_new):
    """Build CALENDAR_PLACEMENTS calendars (each with new events placed
    differently), run the win-rate sim on each, and pool the results.

    Returns
    -------
    pooled_peaks      : ((CALENDAR_PLACEMENTS*TRIALS) x n_assets)
                        peak concurrent demand across all placements & trials.
    pooled_asset_days : ((CALENDAR_PLACEMENTS*TRIALS) x n_assets)
                        billable asset-days across all placements & trials.
    """
    pooled_peaks, pooled_asset_days, pooled_signed = [], [], []
    for _ in range(CALENDAR_PLACEMENTS):
        s, e, q, w = add_new_events(start, end, qty, win_rate, n_new)
        peaks, asset_days, signed = simulate_demand(s, e, q, w)
        pooled_peaks.append(peaks)
        pooled_asset_days.append(asset_days)
        pooled_signed.append(signed)
    return (
        np.concatenate(pooled_peaks), 
        np.concatenate(pooled_asset_days), 
        np.concatenate(pooled_signed)
    )

### Complete Timeline Simulation

In [ ]:
def run(start, end, qty, win_rate, commit_mode=True):
    """Roll the calendar forward YEAR years, sizing fleet and capex each year.

    Each year: grow the event count, re-derive the peak-demand distribution from
    freshly placed calendars, size owned fleet to the coverage percentile, and
    record the capex needed to close any gap. Fleet ratchets up only — it is
    never sold down — so capex appears only when a re-derived peak breaches the
    fleet already held.

    Parameters
    ----------
    start, end, qty, win_rate : the Year-1 (real) event calendar.
    commit_mode : bool — how this year's new events are frozen into the spine
                  for the next year (passed through to add_new_events).

    Returns
    -------
    results : list of per-year dicts, each containing:
        year       : year index (1-based)
        new_events : whole new events added this year
        peaks      : (placements*trials, n_assets) pooled peak distribution
        asset_days : (placements*trials, n_assets) pooled billable asset-days
        required   : (n_assets,) peak at the coverage percentile THIS year
        capex      : (n_assets,) units bought this year (max(0, required - held))
        fleet_held : (n_assets,) cumulative owned fleet AFTER this year (ratchet)
    """
    count = len(start)
    fleet_held = np.zeros(len(ASSETS))    # cumulative owned fleet, ratchets up only
    results = []

    for yr in range(1, YEAR + 1):
        # How many whole new events this year. Y1 is the real calendar -> none added.
        n_new = 0 if yr == 1 else round(count * (1 + GROWTH)) - round(count)

        # Pooled peak & asset-day distributions across many placements x trials.
        peaks, asset_days, signed = simulate_year(start, end, qty, win_rate, n_new)

        # Size fleet to the per-asset coverage percentile; capex closes the gap.
        # .diagonal() picks each asset's own percentile from the (q x asset) grid.
        required = np.ceil(np.quantile(peaks, COVERAGE_PCT, axis=0)).diagonal()
        capex = np.maximum(0, required - fleet_held)      # only buy the shortfall
        fleet_held = np.maximum(fleet_held, required)     # ratchet: never sell down

        # mean and median of signed events
        signed = {
            "mean" : np.mean(signed.sum(axis=1)), 
            "median" : np.median(signed.sum(axis=1))
        }        

        results.append({
            "year": yr,
            "new_events": n_new,
            "peaks": peaks,
            "asset_days": asset_days,
            "required": required,
            "capex": capex,
            "fleet_held": fleet_held.copy(),   # snapshot AFTER this year (copy! it mutates)
            "signed": signed
        })

        # Commit this year's new events into the spine for the next year.
        if n_new:
            start, end, qty, win_rate = add_new_events(
                start, end, qty, win_rate, n_new, commit_mode)
        count = round(count * (1 + GROWTH))

    _print(results)
    return results


def _print(results):
    """Pretty-print each year: peak percentiles + fleet sizing, then asset-days.

    Peak block is read at HIGH percentiles (a coverage decision against a tail);
    asset-days block at CENTRAL measures (an expected volume). 'required' is this
    year's sized peak; 'fleet held' is cumulative owned fleet (the ratchet).
    """
    pct = lambda p, arr: np.percentile(arr, p, axis=0)
    for r in results:
        print(f"\n=== YEAR {r['year']}  (+{r['new_events']} new events) ===") 
        print("            " + "".join(f"{a:>11}" for a in ASSETS))

        # --- PEAK (sizes the fleet): read a high percentile ---
        print("  -- peak concurrent demand --")
        for p in (50, 75, 85, 95):
            print(f"  P{p:<4}     " + "".join(f"{x:>11.0f}" for x in pct(p, r["peaks"])))
        print("  required  " + "".join(f"{x:>11.0f}" for x in r["required"]))    # this year
        print("  held      " + "".join(f"{x:>11.0f}" for x in r["fleet_held"]))  # cumulative
        print("  buy       " + "".join(f"{x:>11.0f}" for x in r["capex"]))

        # --- ASSET-DAYS (drives revenue): read a central measure ---
        print("  -- billable asset-days --")
        print("  mean      " + "".join(f"{x:>11.0f}" for x in r["asset_days"].mean(axis=0)))
        print("  median    " + "".join(f"{x:>11.0f}" for x in pct(50, r["asset_days"])))

In [ ]:
results = run(start, end, qty, win_rate)

## Excel Output

In [ ]:
import pandas as pd

def export_tidy_to_excel(results, path, sheet="Simulation"):
    rows = []

    for r in results:
        p50 = np.percentile(r["peaks"], 50, axis=0)
        p75 = np.percentile(r["peaks"], 75, axis=0)
        p85 = np.percentile(r["peaks"], 85, axis=0)
        p95 = np.percentile(r["peaks"], 95, axis=0)
        asset_days_mean = r["asset_days"].mean(axis=0)
        asset_days_median = np.percentile(r["asset_days"], 50, axis=0)

        for i, asset in enumerate(ASSETS):
            rows.append(
                {
                    "Year": r["year"],
                    "New Events": r["new_events"],
                    "Asset": asset,
                    "50%": round(p50[i]),
                    "75%": round(p75[i]),
                    "85%": round(p85[i]),
                    "95%": round(p95[i]),
                    "Asset Days (Mean)": round(asset_days_mean[i]),
                    "Asset Days (Median)": round(asset_days_median[i]),
                    "Signed (Mean)": r["signed"]['mean'],
                    "Signed (Median)": r["signed"]['median']
                }
            )

    df = pd.DataFrame(rows)

    with pd.ExcelWriter(path, engine="openpyxl", mode="a",
                        if_sheet_exists="replace") as writer:
        df.to_excel(writer, sheet_name=sheet, index=False)

    return df

In [ ]:
df = export_tidy_to_excel(
    results,
    "Qubes' Model - v4.xlsx",
    sheet="Simulation"
)